# Point-SAM forest training on Google Colab

Clones this repo, installs Point-SAM + CUDA ops, downloads the Wytham data, builds crops, then runs the smoke-test training notebook.

**Before anything else:** **Runtime → Change runtime type → GPU** (T4 is fine for a smoke test).

Full write-up: [`docs/colab.md`](../docs/colab.md).

## 0. Options

Set these once, then run the rest top to bottom. Use Drive if you want data and checkpoints to survive a disconnect.

In [12]:
REPO_URL = "https://github.com/koshiyomi/pointsam-training.git"
BRANCH = "jianwei-branch"  # branch that contains scripts/setup_colab.py
USE_DRIVE = True  # keep repo/data/weights on Drive across new Colab VMs
FORCE_REFRESH = True  # True only after you push unzip fixes and want Drive repo updated
DOWNLOAD_DATA = True  # auto-skipped if crops already exist on Drive
PREPARE_DATA = True  # auto-skipped if crops already exist on Drive
TRAIN_CONFIG = "configs/experiment/smoke_colab.yaml"  # then isprs_lora_colab.yaml

## 1. Clone (or reopen) the repo

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

assert Path("/content").exists(), "This notebook expects Google Colab (/content)."

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive/pointsam-training")
else:
    root = Path("/content/pointsam-training")

setup_script = root / "scripts" / "setup_colab.py"

def _run(cmd):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

def _refresh_git_checkout():
    print(f"Refreshing {root} to origin/{BRANCH}", flush=True)
    _run(["git", "-C", str(root), "remote", "set-url", "origin", REPO_URL])
    _run(["git", "-C", str(root), "fetch", "--depth", "1", "origin", BRANCH])
    _run(["git", "-C", str(root), "checkout", "-B", BRANCH, f"origin/{BRANCH}"])
    _run(["git", "-C", str(root), "reset", "--hard", f"origin/{BRANCH}"])

def _reclone():
    bak = root.with_name(root.name + "_stale_bak")
    if bak.exists():
        shutil.rmtree(bak)
    if root.exists():
        root.rename(bak)
        print(f"Moved aside → {bak}", flush=True)
    root.parent.mkdir(parents=True, exist_ok=True)
    _run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(root)])

if not (root / "src" / "treeseg").is_dir():
    root.parent.mkdir(parents=True, exist_ok=True)
    _run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(root)])
elif FORCE_REFRESH or not setup_script.is_file():
    if (root / ".git").exists():
        _refresh_git_checkout()
    else:
        _reclone()
else:
    print("repo already present:", root)

os.chdir(root)
sys.path.insert(0, "src")
if not setup_script.is_file():
    raise FileNotFoundError(
        f"{setup_script} still missing after refresh. Check REPO_URL/BRANCH, or delete "
        f"{root} in Drive and re-run this cell."
    )
print("cwd:", os.getcwd())
print("setup script:", setup_script)

## 2. Install Point-SAM, weights, and (optional) data

**First Colab VM:** slow — compiles `torkit3d`, may download ~3.2 GB and build crops.

**Later VMs (with Drive):** much faster — reuses Drive crops/weights, installs a cached `torkit3d` wheel from `MyDrive/pointsam-colab-cache/`, and skips data if crops exist. Safe to re-run.

In [6]:
# Drive must already be mounted in step 1. Do not pass --drive here: setup runs in a
# subprocess with no Colab kernel, so google.colab.drive.mount fails.
#
# Zenodo zips often use compression Python zipfile cannot read (NotImplementedError).
# Prefer system unzip before setup_colab downloads/prepares.
from pathlib import Path
import subprocess

raw = Path("data/raw/isprs_wytham")
zip_path = raw / "wytham.zip"
plys = [raw / f"wytham_{s}.ply" for s in ("train", "val", "test")]
if zip_path.is_file() and not all(p.is_file() for p in plys):
    print(f"Extracting {zip_path} with system unzip...", flush=True)
    raw.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["unzip", "-o", "-q", str(zip_path), "-d", str(raw)])
    # flatten nested Zenodo layout if needed
    for p in plys:
        if p.is_file():
            continue
        hits = list(raw.rglob(p.name))
        if hits:
            hits[0].replace(p)

flags = []
if DOWNLOAD_DATA:
    flags.append("--download-data")
if PREPARE_DATA:
    flags.append("--prepare-data")

!python scripts/setup_colab.py {" ".join(flags)}

Extracting data/raw/isprs_wytham/wytham.zip with system unzip...
+ git -C /content/drive/MyDrive/pointsam-training/thirdparty/Point-SAM checkout -q 25f4fd9675479f3c951e830771595855ff7ed466
+ git -C /content/drive/MyDrive/pointsam-training/thirdparty/Point-SAM submodule update --init -q third_party/torkit3d
Point-SAM commit: 25f4fd967547
+ /usr/bin/python3 -m pip install -q -r /content/drive/MyDrive/pointsam-training/requirements.txt
+ /usr/bin/python3 -m pip install -q ninja
torkit3d already importable; skip rebuild
weights already present (1244 MB)
Wytham PLYs already in /content/drive/MyDrive/pointsam-training/data/raw/isprs_wytham
+ /usr/bin/python3 /content/drive/MyDrive/pointsam-training/scripts/prepare_data.py configs/data/isprs_wytham_colab.yaml
Traceback (most recent call last):
  File "/usr/lib/python3.13/subprocess.py", line 1281, in wait
    return self._wait(timeout=timeout)
           ~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 2086, in _wa

## 3. Smoke-test train

Runs `notebooks/02_train.ipynb` with the config above. After smoke works, change `TRAIN_CONFIG` to `configs/experiment/isprs_lora.yaml` and re-run this cell (or open `02_train` interactively).

In [ ]:
import os
import torch

os.environ["TREESEG_CONFIG"] = TRAIN_CONFIG
print("config:", TRAIN_CONFIG)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Enable a GPU runtime, then Runtime → Restart session and re-run from setup."

%run notebooks/02_train.ipynb

## Next

- Checkpoints and curves land in `results/<run>/` (and under Drive if `USE_DRIVE`).
- Score a run with `notebooks/03_benchmark.ipynb` after setting `TREESEG_CONFIG` to a benchmark YAML.
- TreeX clicks need the separate ICE TreeX install; stem / label clicks work here without it.